# Project Template: Web Scrape + API → SQL → Business Insights

**Reusable template** based on the *NYC Restaurant Inspections Capstone* pattern.

Use this notebook as a starting point for any project that:
1. Scrapes or bulk-downloads a public dataset (Dataset A)
2. Collects a second dataset from a REST API, with pagination (Dataset B)
3. Cleans and standardizes both datasets so they share a join key
4. Loads both into a SQL (SQLite) database
5. Writes analytical SQL queries to answer specific business questions
6. Visualizes the results and turns them into recommendations

Search for `# TODO:` comments — that's everywhere you need to plug in your own project's specifics. Everything else is boilerplate you can reuse as-is.

---
## Table of Contents
- [Step 0: Define Your Business Problem](#step-0)
- [Step 1: Load Modules](#step-1)
- [Step 2: Acquire Dataset A (Scrape / Bulk Download)](#step-2)
- [Step 3: Acquire Dataset B (API Collection)](#step-3)
- [Step 4: Clean & Standardize Both Datasets](#step-4)
- [Step 5: Load Into a SQL Database](#step-5)
- [Step 6: Query for Insights](#step-6)
- [Step 7: Visualize Findings](#step-7)
- [Step 8: Summarize Recommendations](#step-8)


<a id="step-0"></a>
## Step 0: Define Your Business Problem

Before writing any code, fill this out. It keeps the rest of the notebook focused.

- **Client / stakeholder:** TODO — who will use this analysis?
- **Primary entity of analysis:** TODO — e.g., a restaurant, a store, a listing (this becomes your join key)
- **Business questions (3-5):**
  1. TODO
  2. TODO
  3. TODO


<a id="step-1"></a>
## Step 1: Load Modules

Standard imports for this pipeline. Add domain-specific libraries (e.g., `geopandas` for mapping) as needed.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sqlite3
import requests
import os
from dotenv import load_dotenv

sns.set_theme(style="whitegrid")


<a id="step-2"></a>
## Step 2: Acquire Dataset A (Scrape / Bulk Download)

Get your first dataset — typically a public regulatory, operational, or open-data table.

**Common approaches:**
- `pd.read_html(url)` for an HTML table
- `requests` + `BeautifulSoup` for more complex scraping
- Direct file download (`pd.read_csv(url)`) if the source publishes flat files


In [ ]:
# TODO: replace with your Dataset A source URL
url_a = "https://example.com/your-dataset-a.html"

# TODO: choose the right retrieval method for your source
tables = pd.read_html(url_a)

print(f"Found {len(tables)} table(s).")


In [ ]:
# TODO: confirm which table index has your data, and save it
dataset_a = tables[0]

dataset_a.head()


### Data Dictionary

TODO: link or describe what each important column in Dataset A represents.

<a id="step-3"></a>
## Step 3: Acquire Dataset B (API Collection)

Collect your second dataset via a REST API. This pattern handles pagination so you can
collect large datasets across many requests.

**Directions:**
1. Load environment variables and get your API key from `os.getenv()`.
2. Define request parameters (key, limit, offset).
3. Loop through pages, appending each page's results, and adjusting the offset.
4. Handle failed requests gracefully — log and continue rather than crash.


In [ ]:
# TODO: replace with your Dataset B API endpoint
url_b = "https://example.com/api/your_endpoint"

load_dotenv()

# TODO: set your API key's environment variable name
API_KEY = os.getenv("YOUR_API_KEY_ENV_VAR")

# TODO: adjust limit / page size for your API
PAGE_SIZE = 1000
# TODO: adjust number of pages needed to cover your target record count
N_PAGES = 10

params = {
    "api_key": API_KEY,
    "limit": PAGE_SIZE,
    "offset": 0,
}

errors = []
partial_frames = []

for i in range(N_PAGES):
    params["offset"] = i * PAGE_SIZE
    response = requests.get(url_b, params=params)

    if response.status_code == 200:
        # TODO: adjust this to match your API's response shape
        response_data = response.json().get("data", response.json())
        partial_frames.append(pd.DataFrame(response_data))
    else:
        print(f"Failed to retrieve page {i}")
        errors.append(i)

dataset_b = pd.concat(partial_frames, ignore_index=True) if partial_frames else pd.DataFrame()

if not errors:
    print("Dataset B retrieved successfully.")
else:
    print(f"{len(errors)} page(s) failed: {errors}")


In [ ]:
dataset_b.head()


<a id="step-4"></a>
## Step 4: Clean & Standardize Both Datasets

This is the step most likely to make or break your join later. Follow this checklist:

- [ ] Remove duplicate records
- [ ] Standardize the join-key text field (uppercase, strip special characters, collapse double spaces)
- [ ] Convert date/time columns to `datetime`
- [ ] Convert any categorical scale (e.g., `$` symbols) to a consistent numeric scale
- [ ] Extract any composite fields needed for the join (e.g., ZIP code from an address string)
- [ ] Cast every column to its correct dtype
- [ ] Validate logical/geographic bounds and drop out-of-scope rows


In [ ]:
# --- Clean Dataset A ---
dataset_a_clean = dataset_a.copy()

# TODO: convert relevant date columns
# dataset_a_clean["DATE_COL"] = pd.to_datetime(dataset_a_clean["DATE_COL"])

dataset_a_clean.dtypes


In [ ]:
# --- Clean Dataset B ---
dataset_b_clean = dataset_b.copy()
dataset_b_clean = dataset_b_clean.drop_duplicates()

# TODO: standardize your join-key text column, e.g. a name field
# dataset_b_clean["name"] = dataset_b_clean["name"].str.upper()
# special_char_pattern = r"[^A-Z0-9 ]"
# dataset_b_clean["name"] = dataset_b_clean["name"].str.replace(special_char_pattern, "", regex=True)
# dataset_b_clean["name"] = dataset_b_clean["name"].str.replace("  ", " ")

# TODO: convert any categorical scale to numeric, in the correct replacement order
# possible_levels = ["$$$$$", "$$$$", "$$$", "$$", "$", " - No ratings yet"]
# for i in range(len(possible_levels)):
#     dataset_b_clean["price_level"] = dataset_b_clean["price_level"].str.replace(possible_levels[i], str(5 - i))

# TODO: extract a join key like zip code from a free-text field
# dataset_b_clean["zip_code"] = dataset_b_clean["address"].str.extract(r"(\d{5})")

dataset_b_clean.head()


In [ ]:
# --- Data quality checks ---
print("Dataset A dtypes:\n", dataset_a_clean.dtypes, "\n")
print("Dataset B dtypes:\n", dataset_b_clean.dtypes, "\n")

print("Dataset A nulls:\n", dataset_a_clean.isnull().sum(), "\n")
print("Dataset B nulls:\n", dataset_b_clean.isnull().sum())


In [ ]:
# TODO: drop out-of-scope records, e.g. join keys outside a valid range
# dataset_b_clean = dataset_b_clean[(dataset_b_clean["zip_code"] >= 10001) & (dataset_b_clean["zip_code"] <= 11697)]


<a id="step-5"></a>
## Step 5: Load Into a SQL Database

Persist both cleaned tables into SQLite so you can query them with SQL — more efficient and shareable than keeping everything in-memory.

In [ ]:
# TODO: name your database file
DB_NAME = "project_data.db"

connection = sqlite3.connect(DB_NAME)

# TODO: name your tables
dataset_a_clean.to_sql("table_a", connection, if_exists="replace", index=False)
dataset_b_clean.to_sql("table_b", connection, if_exists="replace", index=False)

connection.close()
print(f"Data saved to {DB_NAME}")


<a id="step-6"></a>
## Step 6: Query for Insights

Reconnect to the database and write queries that directly answer your business questions from Step 0.

**Query patterns to reuse:**
1. Aggregate a metric by one categorical dimension (`GROUP BY` + `COUNT` + `AVG`)
2. Join the two tables on your shared key, filtering out missing values
3. Compute a rate/percentage with a conditional aggregate (`AVG(CASE WHEN ... THEN 1 ELSE 0 END) * 100`)


In [ ]:
connection = sqlite3.connect(DB_NAME)

# sanity check
pd.read_sql_query("SELECT * FROM table_b LIMIT 1", connection)


In [ ]:
# TODO: Pattern 1 — metric by category
query_by_category = """
SELECT category_column,
       COUNT(*) as total_records,
       AVG(metric_column) as avg_metric
FROM table_a
GROUP BY category_column
ORDER BY avg_metric
"""
# by_category_df = pd.read_sql_query(query_by_category, connection)
# by_category_df.head()


In [ ]:
# TODO: Pattern 2 — join across tables
query_joined = """
SELECT DISTINCT a.key_column,
                a.metric_column,
                b.other_column
FROM table_a a
JOIN table_b b ON a.join_key = b.join_key
WHERE a.metric_column IS NOT NULL
ORDER BY a.metric_column DESC
"""
# joined_df = pd.read_sql_query(query_joined, connection)
# joined_df.head()


In [ ]:
# TODO: Pattern 3 — rate/percentage with conditional aggregate
query_rate = """
SELECT category_column,
       COUNT(*) as total_records,
       AVG(CASE WHEN metric_column <= 13 THEN 1 ELSE 0 END) * 100 as pass_rate,
       AVG(metric_column) as avg_metric
FROM table_a
WHERE metric_column IS NOT NULL
GROUP BY category_column
ORDER BY pass_rate DESC
"""
# rate_df = pd.read_sql_query(query_rate, connection)
# rate_df.head()


In [ ]:
connection.close()


<a id="step-7"></a>
## Step 7: Visualize Findings

Reuse these chart patterns for the query results above.

In [ ]:
# TODO: bar chart — metric by category
# plt.figure(figsize=(8, 5))
# sns.barplot(data=by_category_df, x="avg_metric", y="category_column")
# plt.title("TODO: descriptive chart title")
# plt.tight_layout()
# plt.show()


In [ ]:
# TODO: box plot — distribution comparison across a segment
# plt.figure(figsize=(8, 5))
# sns.boxplot(data=joined_df, x="segment_column", y="metric_column")
# plt.title("TODO: descriptive chart title")
# plt.tight_layout()
# plt.show()


<a id="step-8"></a>
## Step 8: Summarize Recommendations

For each business question from Step 0, state the finding and the recommended action.

| Business Question | Finding | Recommendation |
|---|---|---|
| TODO | TODO | TODO |
| TODO | TODO | TODO |
| TODO | TODO | TODO |

---
*Template based on the C4 Capstone: NYC Restaurant Inspections Analysis pipeline.*
